In [1]:
# !pip install getml pyarrow relbench

In [2]:
import getml
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("rel-hm", download=True)
task = get_task("rel-hm", "user-churn", download=True)

getml.engine.launch(in_memory=False)

getml.set_project("hm-user-final")

getML Engine is already running.


Output()

Connected to project 'hm-user-final'.

In [3]:
def load_df(path: str, name: str, roles: getml.data.Roles) -> getml.data.DataFrame:
    """
    Load a DataFrame from a parquet file and save it in getml native format to
    disk if it does not exist yet.
    """
    # if getml.data.exists(name):
        # return getml.data.load_data_frame(name)

    df = getml.data.DataFrame.from_parquet(
        path,
        name=name,
        roles=roles,
    )
    df.save()
    return df


population_roles = getml.data.Roles(
    join_key=["customer_id"],
    target=["churn"],
    time_stamp=["timestamp"],
)

subsets = ("train", "test", "val")
populations = {}
for subset in subsets:
    populations[subset] = load_df(
        f"{dataset.cache_dir}/tasks/user-churn/{subset}.parquet",
        subset,
        population_roles,
    )
    # populations[subset]["reference_date"] = dataset.test_timestamp.to_datetime64()
    # populations[subset].set_role(["reference_date"], getml.data.roles.time_stamp)

customer_roles = getml.data.Roles(
    join_key=["customer_id"],
    numerical=["age"],
    categorical=[
        "club_member_status",
        "fashion_news_frequency",
        "postal_code",
        # "FN", "Active"
    ],
)

customer = load_df(
    f"{dataset.cache_dir}/db/customer.parquet", "customer", customer_roles
)
customer.set_unit(["FN", "Active", "postal_code"], "comparison only")

transaction_roles = getml.data.Roles(
    join_key=["article_id", "customer_id"],
    time_stamp=["t_dat"],
    numerical=["price"],
    categorical=["sales_channel_id"],
)

transaction = load_df(
    f"{dataset.cache_dir}/db/transactions.parquet", "transaction", transaction_roles
)

article_roles = getml.data.Roles(
    join_key=["article_id"],
    categorical=[
        # "product_type_name",
        # "graphical_appearance_name",
        # "colour_group_name",
        # "perceived_colour_value_name",
        # "perceived_colour_master_name",
        # "section_no",
        # "index_name",
        # "index_group_name",
        "product_group_name",
        "department_name",
        "garment_group_name",
    ],
)

article = load_df(f"{dataset.cache_dir}/db/article.parquet", "article", article_roles)

dm = getml.data.DataModel(population=populations["train"].to_placeholder())
dm.add(getml.data.to_placeholder(customer, transaction, article))
dm.population.join(
    dm.customer, on="customer_id", relationship=getml.data.relationship.many_to_one
)
dm.population.join(
    dm.transaction, on="customer_id", time_stamps=("timestamp", "t_dat")
)
dm.transaction.join(
    dm.article, on="article_id", relationship=getml.data.relationship.many_to_one
)

container = getml.data.Container(**populations)
container.add(customer, transaction, article)



In [4]:
pipe = getml.Pipeline(
    tags=["task: item-churn; model:fastprop, seasonal, mapping"],
    data_model=dm,
    preprocessors=[getml.preprocessors.Seasonal(), getml.preprocessors.Mapping()],
    feature_learners=[
        getml.feature_learning.FastProp(num_threads=24)
    ],
    # include_categorical=True,
    predictors=[getml.predictors.XGBoostClassifier(n_jobs=24)],
    loss_function=getml.feature_learning.loss_functions.CrossEntropyLoss,
)

In [5]:
pipe.check(container.train)
pipe.fit(container.train)

Checking data model...

Output()

OK.

Checking data model...

Output()

OK.

Output()

Trained pipeline.

Time taken: 0:10:23.898651.



Pipeline(data_model='train',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='CrossEntropyLoss',
         peripheral=['article', 'customer', 'transaction'],
         predictors=['XGBoostClassifier'],
         preprocessors=['Seasonal', 'Mapping'],
         share_selected_features=0.5,
         tags=['task: user-churn; model:simple;', 'container-PcbmNJ'])

In [6]:
pipe.score(container.val)
pipe.score(container.test)
pipe.scores

Output()

Output()

,date time,set used,target,accuracy,auc,cross entropy
0,2024-12-18 19:05:08,train,churn,0.8258,0.7,0.4307
1,2024-12-18 19:05:30,val,churn,0.8203,0.7062,0.439
2,2024-12-18 19:05:51,test,churn,0.8321,0.7014,0.4281
